In [ ]:
import pickle
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# Paths

INTERACTION_PATH = ".txt file of user item interaction data(json)"
ID2TITLE_PATH = "text_name_dictionary file(json)"
OUTPUT_PATH = "weight saving output file(.pt extensiont"

          
BATCH_SIZE = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Load interaction file to get item universe

items = set()
with open(INTERACTION_PATH, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            _, item = map(int, parts[:2])
            items.add(item)

items = sorted(items)

# Map raw item id -> continuous index 
i2id = {i: idx for idx, i in enumerate(items)}
num_items = len(i2id)

print(f"Number of items: {num_items}")


with open(ID2TITLE_PATH, "rb") as f:
    id2title = pickle.load(f)


item_texts = []
for raw_item_id in items:
    text = id2title.get(raw_item_id, "")
    if text is None or len(str(text).strip()) == 0:
        text = "unknown item"
    item_texts.append(str(text))


sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)


embs = []
for i in tqdm(range(0, num_items, BATCH_SIZE)):
    batch_texts = item_texts[i:i+BATCH_SIZE]
    batch_emb = sbert.encode(
        batch_texts,
        convert_to_tensor=True,
        device=device,
        normalize_embeddings=True
    )
    embs.append(batch_emb)

item_embs = torch.cat(embs, dim=0)   # [num_items, 384]

print(item_embs.shape)
item_embs = item_embs.cpu()
torch.save(item_embs, OUTPUT_PATH)

print(f"Saved SBERT item embeddings to: {OUTPUT_PATH}")
